# TTL-aligned flight times for Dataset_3_29959_240403, clusters 27 vs 28

Goal: get the TTL-aligned start times for each flight in clusters 27 and 28, so we can navigate to the corresponding frames in the overhead fisheye video (20 fps).

In [21]:
import sys
sys.path.insert(0, '/home/alowet/Documents/code/digitalbat')

import numpy as np
import pandas as pd
from scipy.io import loadmat
import glob

from digitalbat.utility.utils import FlightClusters

SESSION_NAME = 'Dataset_3_29959_240403'
PARENT_FOLDER = '/mnt/s5/Adam/Angelo/NP_datasets/'
MIN_FLIGHT_DUR = 1.0  # seconds

# Load raw behavior
bhv_file = glob.glob(PARENT_FOLDER + SESSION_NAME + '/Extracted_Behavior_*.mat')[0]
bhv = loadmat(bhv_file)

r = bhv['r']
t_bhv = bhv['t'][0, :]  # Dataset_3 uses row vector
bflying = bhv['bflying'][:, 0].astype(int)
Fs = float(bhv['Fs'][0, 0])

print(f"Session: {SESSION_NAME}")
print(f"Fs = {Fs} Hz, duration = {t_bhv[-1] - t_bhv[0]:.1f} s")
print(f"t_bhv range: [{t_bhv[0]:.3f}, {t_bhv[-1]:.3f}] s")

Session: Dataset_3_29959_240403
Fs = 120.0 Hz, duration = 4722.7 s
t_bhv range: [-70.621, 4652.088] s


In [22]:
# Run flight clustering (same params as investigate_similar_trajectories notebook)
fc = FlightClusters(r_vector=r, f_vector=bflying, Fs=Fs,
                    frechet_clus=True, alpha=0.8)
fc_results = fc.compute()

# Segment flights manually to get start/stop frame indices
starts = np.where(np.diff(bflying) > 0)[0] + 1
stops = np.where(np.diff(bflying) < 0)[0]
if bflying[0] > 0:
    starts = np.concatenate([[0], starts])
if bflying[-1] > 0:
    stops = np.concatenate([stops, [len(bflying) - 1]])
n = min(len(starts), len(stops))
starts, stops = starts[:n], stops[:n]
durs = (stops - starts) / Fs
keep = durs >= MIN_FLIGHT_DUR
starts, stops = starts[keep], stops[keep]

cluster_ids = fc_results['id']
print(f"Total flights (>= {MIN_FLIGHT_DUR}s): {len(starts)}")
print(f"Cluster IDs found: {np.unique(cluster_ids)}")

Total flights (>= 1.0s): 131
Cluster IDs found: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37]


In [23]:
# t_bhv=0 corresponds to TTL start = video frame 0
VIDEO_FPS = 20

rows = []
for cid in [27, 28]:
    flight_indices = np.where(cluster_ids == cid)[0]
    for fi in flight_indices:
        t_start = t_bhv[starts[fi]]
        t_stop = t_bhv[stops[fi]]
        rows.append(dict(
            cluster=cid,
            flight_idx=fi,
            t_start_s=t_start,
            t_stop_s=t_stop,
            duration_s=t_stop - t_start,
            video_frame_start=int(t_start * VIDEO_FPS),
            video_frame_stop=int(t_stop * VIDEO_FPS),
        ))

df = pd.DataFrame(rows)
print(f"Cluster 27: {(df.cluster == 27).sum()} flights")
print(f"Cluster 28: {(df.cluster == 28).sum()} flights")
print()
df

Cluster 27: 36 flights
Cluster 28: 3 flights



,cluster,flight_idx,t_start_s,t_stop_s,duration_s,video_frame_start,video_frame_stop
0,27,3,252.970833,254.995833,2.025000,5059,5099
1,27,5,312.804167,314.754165,1.949999,6256,6295
2,27,7,359.762499,361.754166,1.991667,7195,7235
3,27,9,397.204167,399.120833,1.916666,7944,7982
4,27,11,439.587500,441.520833,1.933333,8791,8830
5,27,13,475.329166,477.337499,2.008333,9506,9546
6,27,16,538.787501,540.862501,2.075000,10775,10817
7,27,19,613.320833,615.212500,1.891666,12266,12304
8,27,23,676.437500,678.554167,2.116667,13528,13571
9,27,25,715.970833,717.954167,1.983334,14319,14359


In [24]:
# Match videos to TTL time ranges using TTL pulse spacing and video durations
import os, subprocess

# Load TTL pulses from any probe's Kilosort output
su_file = glob.glob(PARENT_FOLDER + SESSION_NAME + '/SU_kilosort4_outdir_probe1.mat')[0]
su = loadmat(su_file)
ttl_local = su['out']['local_ttl_timestamps_usec'][0, 0].ravel() / 1e6  # seconds on local clock

# Check for a gap in TTL pulses (would indicate two recording blocks = two videos)
ttl_intervals = np.diff(ttl_local)
median_interval = np.median(ttl_intervals)
gap_mask = ttl_intervals > 10 * median_interval  # anything >>3s is a gap
gap_indices = np.where(gap_mask)[0]

print(f"TTL pulses: {len(ttl_local)}, interval ~{median_interval:.2f} s")
print(f"TTL range: {ttl_local[0]:.1f} - {ttl_local[-1]:.1f} s (local clock)")

if len(gap_indices) > 0:
    for gi in gap_indices:
        gap_dur = ttl_intervals[gi]
        print(f"\nGap at pulse {gi}: {gap_dur:.1f} s ({gap_dur/60:.1f} min)")
        print(f"  Block 1: pulses 0-{gi}, local time {ttl_local[0]:.1f} - {ttl_local[gi]:.1f} s, "
              f"duration {ttl_local[gi] - ttl_local[0]:.1f} s ({(ttl_local[gi] - ttl_local[0])/60:.1f} min)")
        print(f"  Block 2: pulses {gi+1}-{len(ttl_local)-1}, local time {ttl_local[gi+1]:.1f} - {ttl_local[-1]:.1f} s, "
              f"duration {ttl_local[-1] - ttl_local[gi+1]:.1f} s ({(ttl_local[-1] - ttl_local[gi+1])/60:.1f} min)")
else:
    print(f"\nNo gap found — single continuous block, duration {ttl_local[-1] - ttl_local[0]:.1f} s")

# Get video durations
video_dir = '/media/alowet/angelo2/Neuropixel_Data_AFWF/Raw/Dataset_3/240403/Cameras/240403/'
fisheye_videos = sorted(glob.glob(video_dir + 'Basler_acA2440-35um__23418165__*.mp4'))

print(f"\nVideo durations:")
video_info = []
for v in fisheye_videos:
    result = subprocess.run(
        ['ffprobe', '-v', 'quiet', '-show_entries', 'format=duration',
         '-of', 'csv=p=0', v],
        capture_output=True, text=True)
    dur = float(result.stdout.strip())
    fname = os.path.basename(v)
    video_info.append((v, fname, dur))
    print(f"  {fname}: {dur:.1f} s ({dur/60:.1f} min)")

TTL pulses: 1550, interval ~3.00 s
TTL range: 39.4 - 4686.4 s (local clock)

No gap found — single continuous block, duration 4647.0 s

Video durations:
  Basler_acA2440-35um__23418165__20240403_150640929.mp4: 4620.6 s (77.0 min)
  Basler_acA2440-35um__23418165__20240403_163414124.mp4: 4648.1 s (77.5 min)


In [25]:
# Map TTL blocks to videos by matching durations, then assign flights to videos
# t_bhv=0 is TTL-aligned; local_ttl starts at ~39s on the probe clock.
# The offset between local clock and TTL-aligned time:
#   t_bhv ≈ local_time - ttl_local[0]  (first TTL pulse = t_bhv ≈ 0)

# Build block boundaries in TTL-aligned time (t_bhv space)
ttl_offset = ttl_local[0]  # local clock time of first TTL pulse

if len(gap_indices) > 0:
    gi = gap_indices[0]  # use first gap
    block_bounds_ttl = [
        (ttl_local[0] - ttl_offset, ttl_local[gi] - ttl_offset),
        (ttl_local[gi + 1] - ttl_offset, ttl_local[-1] - ttl_offset),
    ]
    block_durs = [b[1] - b[0] for b in block_bounds_ttl]

    # Match blocks to videos by closest duration
    vid_durs = [info[2] for info in video_info]
    print("Block durations (TTL):", [f"{d:.1f}s" for d in block_durs])
    print("Video durations:      ", [f"{d:.1f}s" for d in vid_durs])

    # Assign: for each block, pick the video with closest duration
    block_to_video = {}
    used = set()
    for bi, bd in enumerate(block_durs):
        best_vi = min([i for i in range(len(vid_durs)) if i not in used],
                      key=lambda i: abs(vid_durs[i] - bd))
        block_to_video[bi] = best_vi
        used.add(best_vi)
        print(f"\n  Block {bi+1} ({bd:.1f}s, TTL {block_bounds_ttl[bi][0]:.1f}-{block_bounds_ttl[bi][1]:.1f}s) "
              f"→ {video_info[best_vi][1]} ({vid_durs[best_vi]:.1f}s)")

    # Determine which block the flights fall in, compute video-relative time
    def get_block(t_start):
        for bi, (bstart, bstop) in enumerate(block_bounds_ttl):
            if bstart - 30 <= t_start <= bstop + 30:
                return bi, t_start - bstart
        return -1, -1

    df['block'] = df.t_start_s.apply(lambda t: get_block(t)[0])
    df['video_time_s'] = df.t_start_s.apply(lambda t: get_block(t)[1])
    df['video_frame'] = (df.video_time_s * VIDEO_FPS).astype(int)

    # Print video file per block
    for bi, vi in block_to_video.items():
        cluster_in_block = df.loc[df.block == bi, 'cluster'].unique()
        if len(cluster_in_block):
            print(f"\n  Clusters {sorted(cluster_in_block)} → open: {video_info[vi][1]}")
else:
    # Single block — all flights in one video; pick longest video
    best_vi = max(range(len(video_info)), key=lambda i: video_info[i][2])
    print(f"Single recording block → open: {video_info[best_vi][1]}")
    df['video_time_s'] = df.t_start_s
    df['video_frame'] = (df.t_start_s * VIDEO_FPS).astype(int)

def secs_to_hms(s):
    h, rem = divmod(int(s), 3600)
    m, sec = divmod(rem, 60)
    return f"{h}:{m:02d}:{sec:02d}"

df['video_hms'] = df.video_time_s.apply(secs_to_hms)

df[['cluster', 'flight_idx', 't_start_s', 'duration_s', 'video_time_s', 'video_hms', 'video_frame']]

Single recording block → open: Basler_acA2440-35um__23418165__20240403_163414124.mp4


,cluster,flight_idx,t_start_s,duration_s,video_time_s,video_hms,video_frame
0,27,3,252.970833,2.025000,252.970833,0:04:12,5059
1,27,5,312.804167,1.949999,312.804167,0:05:12,6256
2,27,7,359.762499,1.991667,359.762499,0:05:59,7195
3,27,9,397.204167,1.916666,397.204167,0:06:37,7944
4,27,11,439.587500,1.933333,439.587500,0:07:19,8791
5,27,13,475.329166,2.008333,475.329166,0:07:55,9506
6,27,16,538.787501,2.075000,538.787501,0:08:58,10775
7,27,19,613.320833,1.891666,613.320833,0:10:13,12266
8,27,23,676.437500,2.116667,676.437500,0:11:16,13528
9,27,25,715.970833,1.983334,715.970833,0:11:55,14319
